# Categorização Automática de Demandas de Ouvidoria
### Trabalho final — Deep Learning e PLN (IDP) · Modalidade 2 (NLP no Setor Público)

**Integrantes:** _(preencher)_  
**Data:** _(preencher)_

---

**Problema.** Direcionar automaticamente manifestações de cidadãos para a área responsável
(Saneamento, Iluminação, Trânsito, Saúde, etc.) a partir do texto livre da reclamação.  
Tarefa: **classificação de texto multiclasse** em português do Brasil.

**Hipótese central.** Um modelo de Deep Learning (Transformer com fine-tuning) supera o
baseline clássico (TF-IDF + modelo linear) na métrica **F1-macro**.

> Notebook **orquestrador**: a lógica pesada vive em `src/`; aqui apenas importamos, chamamos e narramos.

## 0. Setup e configuração

Imports gerais, seed e caminhos. Ver `requirements.txt` para o ambiente.

In [ ]:
import sys
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Permite importar de src/ (notebook está em notebooks/, src/ está em ../src/)
RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ))

DATA_RAW = RAIZ / "data" / "raw"
DATA_INTERIM = RAIZ / "data" / "interim"
DATA_PROCESSED = RAIZ / "data" / "processed"
RESULTADOS = RAIZ / "resultados"

for p in (DATA_RAW, DATA_INTERIM, DATA_PROCESSED, RESULTADOS, RESULTADOS / "figuras"):
    p.mkdir(parents=True, exist_ok=True)

print("Ambiente configurado. Seed =", RANDOM_STATE)
print("Raiz do projeto:", RAIZ)

## 1. Coleta de dados

> ⚠️ **Guardrail:** dados devem ser **coletados pela equipe** (scraping/API), não baixados prontos.  
> Antes de rodar scraping ao vivo: confirmar fonte com o professor, checar `robots.txt`,  
> aplicar *delay* de 2s, identificar *user-agent*, anonimizar dados pessoais e gravar log.

**Fonte escolhida:** _(preencher: Fala.BR / Consumidor.gov.br / Portal 156)_  
**Período:** _(preencher: ex. 2025-01-01 a 2026-05-31)_  
**Volume coletado:** _(preencher após execução)_

In [ ]:
from src.coleta.coletor import coletar

# ATENÇÃO: defina dry_run=False apenas após confirmação da equipe.
# dry_run=True verifica o fluxo sem disparar requisições reais.
df_bruto = coletar(
    fonte="consumidor",   # ou "falabr"
    paginas=50,
    delay=2.0,
    dry_run=True,          # alterar para False após confirmação
)

# Carregar dados já coletados (se existirem):
# df_bruto = pd.read_parquet(DATA_RAW / "coleta.parquet")

print(f"Shape: {df_bruto.shape}")
df_bruto.head()

## 2. Rotulagem e concordância entre anotadores

Defina de **5 a 10 categorias** claras. Se a fonte já traz `assunto/problema`, use como rótulo;  
senão, rotule manualmente uma amostra com **dois anotadores** e meça o **kappa de Cohen**  
(diferencial metodológico — ver `docs/decisoes.md` D-001 e D-002).

In [ ]:
from src.preprocessamento.limpeza import calcular_kappa

CATEGORIAS = [
    "Saneamento",
    "Iluminacao",
    "Transito",
    "Saude",
    "Limpeza",
    "Outros",
]

# --- Se a fonte já traz rótulos ---
# df_rotulado = df_bruto.rename(columns={"categoria": "categoria"})
# df_rotulado = df_rotulado[df_rotulado["categoria"].isin(CATEGORIAS)]

# --- Se rotulagem manual: calcular kappa entre dois anotadores ---
# anotador_1 = [...]  # lista de rótulos do anotador A
# anotador_2 = [...]  # lista de rótulos do anotador B
# kappa = calcular_kappa(anotador_1, anotador_2)
# print(f"Kappa de Cohen: {kappa:.3f}")
# # Kappa > 0.6 = concordância substancial (aceitável para publicação)

# Salvar base rotulada:
# df_rotulado.to_parquet(DATA_INTERIM / "rotulado.parquet", index=False)
# print(df_rotulado["categoria"].value_counts())

## 3. Pré-processamento

> Limpeza **agressiva** para o baseline TF-IDF (lowercase, stopwords, lematização spaCy).  
> Limpeza **leve** para o Transformer (preservar contexto — BERT usa a sentença inteira).  

Split estratificado 80/10/10 — estratificar pela categoria (dados desbalanceados).

In [ ]:
from src.preprocessamento.limpeza import limpar_coluna, dividir_dados

df = pd.read_parquet(DATA_INTERIM / "rotulado.parquet")

# Limpeza para BERT (leve)
df = limpar_coluna(df, coluna="texto", modo="bert")
df = df.rename(columns={"texto_limpo": "texto_bert"})

# Limpeza para TF-IDF (agressiva)
df = limpar_coluna(df, coluna="texto", modo="tfidf")
df = df.rename(columns={"texto_limpo": "texto_tfidf"})

# Split estratificado para BERT
X_train_bert, X_val_bert, X_test_bert, y_train, y_val, y_test = dividir_dados(
    df, coluna_texto="texto_bert", coluna_label="categoria", seed=RANDOM_STATE
)

# Split estratificado para TF-IDF (mesmo índice)
X_train_tfidf = df.loc[X_train_bert.index, "texto_tfidf"]
X_val_tfidf = df.loc[X_val_bert.index, "texto_tfidf"]
X_test_tfidf = df.loc[X_test_bert.index, "texto_tfidf"]

print(f"Treino: {len(X_train_bert)} | Validação: {len(X_val_bert)} | Teste: {len(X_test_bert)}")
print("\nDistribuição (treino):")
print(y_train.value_counts())

# Salvar splits
df.to_parquet(DATA_PROCESSED / "dados.parquet", index=False)

## 4. Baseline clássico — TF-IDF + modelo linear

Ponto de comparação **obrigatório**. Rápido e interpretável.  
É contra este F1-macro que o Deep Learning precisa provar valor.

In [ ]:
from src.modelos.baseline import treinar_baseline, avaliar

pipe_baseline, pred_baseline = treinar_baseline(
    X_train_tfidf, y_train, X_test_tfidf, modelo="logistic", seed=RANDOM_STATE
)

metricas_baseline = avaliar(y_test, pred_baseline, nome_modelo="TF-IDF + LogisticRegression")

## 5. Modelo de Deep Learning — fine-tuning de Transformer

Modelo base: **BERTimbau** (`neuralmind/bert-base-portuguese-cased`).  
Se a GPU for limitada, ative `usar_lora=True` para LoRA via `peft`.

In [ ]:
from src.modelos.transformer import treinar_transformer

MODELO_BERT = "neuralmind/bert-base-portuguese-cased"

modelo_dl, pred_transformer, encoder_rotulos = treinar_transformer(
    X_train=X_train_bert,
    y_train=y_train,
    X_val=X_val_bert,
    y_val=y_val,
    X_test=X_test_bert,
    modelo_nome=MODELO_BERT,
    max_length=256,
    epocas=3,
    batch_size=16,
    usar_lora=False,   # True se GPU < 8GB
    seed=RANDOM_STATE,
)

from src.avaliacao.metricas import calcular_metricas
metricas_transformer = calcular_metricas(y_test, pred_transformer, "BERTimbau fine-tuned")

## 6. Avaliação comparativa

> **Priorize F1-macro** (dados desbalanceados). Inclua precisão/revocação por classe  
> e a **matriz de confusão** de cada modelo.

In [ ]:
from src.avaliacao.metricas import comparar_modelos, plotar_f1_por_classe

classes = sorted(y_test.unique().tolist())

resultado_final = comparar_modelos(
    y_test=y_test,
    pred_baseline=pred_baseline,
    pred_transformer=pred_transformer,
    classes=classes,
)

plotar_f1_por_classe(y_test, pred_baseline, pred_transformer, classes)

print("\nGanho absoluto F1-macro:", resultado_final["ganho_f1_macro"])

## 7. Diferencial — categoria prevista × geografia

Cruzar a categoria prevista com o **bairro/CEP** da reclamação e gerar  
um **mapa de calor** de onde o poder público deve alocar recursos por tipo de demanda.

In [ ]:
from src.avaliacao.metricas import gerar_mapa_calor

# Adicionar predições ao DataFrame de teste
df_teste = df.loc[X_test_bert.index].copy()
df_teste["categoria_prevista"] = pred_transformer

# Gerar mapa (requer colunas latitude e longitude no dataset coletado)
# gerar_mapa_calor(
#     df_teste,
#     col_categoria="categoria_prevista",
#     col_lat="latitude",
#     col_lon="longitude",
# )

# Se não houver geo: mapa de calor por categoria vs. UF/município
if "municipio" in df_teste.columns:
    demandas_por_local = (
        df_teste.groupby(["municipio", "categoria_prevista"])
        .size()
        .reset_index(name="contagem")
    )
    print(demandas_por_local.pivot(index="municipio", columns="categoria_prevista", values="contagem"))

## 8. Conclusão e próximos passos

- O Deep Learning superou o baseline? Em quanto (F1-macro)?
- Traduza o ganho em **impacto de serviço público** (triagem mais rápida, menos retrabalho,
  alocação de recursos por região).
- Limitações (tamanho/representatividade da amostra, vieses, categorias ambíguas).
- Próximos passos (mais dados, modelos maiores, humano no loop).

---
**Referências** — ver `docs/referencias.md` (mínimo 5 de domínio + 5 de técnica, ABNT).